<div style="background:linear-gradient(135deg,#1f4e79,#2f80ed 55%,#56ccf2);color:white;padding:26px 30px;border-radius:14px;margin:6px 0">
    <div style="font-size:2em;font-weight:800">
        Simple Statistics
    </div>
    <div style="font-size:1.1em;opacity:.95;margin-top:6px">
        Means, medians and counts of real hourly PM2.5 data
    </div>
</div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    <div style="font-weight:700;font-size:1.08em;margin-bottom:4px">
        ✍️ How to use this notebook...
    </div>
    <ul>
        <li>
            <b>Shift + Enter</b> runs a cell. <b>Do not use “Run All.”</b> Run the two setup cells first, then work down the page in order.
        </li>
        <li>
            Read each passage, then run the code under it and look at what comes out.
        </li>
        <li>
            Change a value and run the cell again. When a cell fails, read the red message from the bottom up.
        </li>
    </ul>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    <div style="font-weight:700;font-size:1.08em;margin-bottom:4px">
        🎯 Outline of this Course Module
    </div>
    <ul>
        <li>
            <b> Section 1: The standard library </b>
        </li>
        <li>
            <b> Section 2: NumPy </b>
        </li>
        <li>
            <b> Section 3: pandas </b>
        </li>
        <li>
            <b> Section 4: Very simple plots </b>
        </li>
    </ul>
</div>

In [ ]:
# Setup, part 1: bring in the tools this notebook uses.
import math
import statistics
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Setup, part 2: find the data file. This tries three places in order.
name = "AMP501_2420560-0.txt"
candidates = [Path("../data") / name, Path("data") / name, Path("/content") / name]  # /content is Google Colab

for path in candidates:
    if path.exists():
        break
else:
    raise FileNotFoundError(
        f"Could not find {name}. Looked in: {[str(p) for p in candidates]}. "
        "Put the file in the course data folder, or upload it to Colab."
    )

print("Using", path)

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 1</span><span style="font-size:1.2em;font-weight:600"> &nbsp;The standard library</span>
    <div style="opacity:.92">
        Open a file, split a line, and take a mean with nothing installed
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do I get a number out of a line of text? Take the first record in the file. To Python it is one long string, with the fields run together and separated by <code>|</code>.
    <p>
        Suppose I call <code>.split('|')</code> on it. I get a list of smaller strings, one per field, numbered from 0. The site is piece 4, the date piece 10, the hour piece 11, and the PM2.5 reading piece 12. But a piece is still text. <code>'0.8'</code> is a zero, a point and an eight. To do arithmetic I wrap it in <code>float()</code>, which turns the text into a number.
    </p>
</div>

In [ ]:
with open(path) as f:
    lines = f.readlines()

line = lines[2]          # lines 0 and 1 are headers, so the first record is line 2
fields = line.split("|")
print(fields[4], fields[10], fields[11], fields[12])
print(float(fields[12]) + 1)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What happens when an hour has no reading? The field is empty, and <code>float('')</code> has no number to make, so Python stops with a <code>ValueError</code>. An empty string counts as false in an <code>if</code>, though, so <code>if fields[12]</code> lets only the hours with a value through.
    <p>
        Loop over the lines, keep the records from site 0002, and collect their readings in a list. A year has 8,760 hours, but this list comes out with 8,744 values.
    </p>
</div>

In [ ]:
values = []
for line in lines:
    if not line.startswith("RD|"):
        continue                      # skip the header and footer lines
    fields = line.split("|")
    if fields[4] == "0002" and fields[12]:
        values.append(float(fields[12]))

print(len(values))

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Do I have to add up 8,744 numbers and divide by hand? No. The <code>statistics</code> module is already part of Python. <code>statistics.mean(values)</code> gives 8.13, and <code>statistics.median(values)</code> gives 6.4. The mean sits above the median because a few very high hours pull it up.
    <p>
        <code>statistics.stdev(values)</code> gives 5.79. That is the square root of the variance, and the <code>math</code> module has <code>math.sqrt</code>, so I can check it: the square root of <code>statistics.variance(values)</code> comes out the same.
    </p>
</div>

In [ ]:
print(statistics.mean(values), statistics.median(values))
print(statistics.stdev(values))
print(math.sqrt(statistics.variance(values)))

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 2</span><span style="font-size:1.2em;font-weight:600"> &nbsp;NumPy</span>
    <div style="opacity:.92">
        Arrays, percentiles, missing values and counting
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Why would NumPy report a standard deviation of 5.7939 when <code>statistics.stdev</code> said 5.7943 for the very same numbers? Put the 8,744 readings in an array with <code>a = np.array(values)</code> and ask. The mean and median match: 8.13 and 6.4.
    <p>
        The two functions are answering different questions. <code>a.std()</code> divides by n, the number of values, as if these hours were the whole population. <code>statistics.stdev</code> divides by n - 1, the usual choice for a sample. Pass <code>ddof=1</code> and NumPy divides by n - 1 as well, and the two agree: 5.7943.
    </p>
</div>

In [ ]:
a = np.array(values)
print(a.mean(), np.median(a))
print(a.std(), a.std(ddof=1))

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How high do the top few hours go? <code>np.percentile(a, 95)</code> returns the value that 95 percent of the readings fall at or below. For this site it is 18.7. Hand it a list of percentiles and it answers for each: the 50th is 6.4, which is the median again, and the 99th is 27.985.
    <p>
        The single highest hour, <code>a.max()</code>, is 62.0, more than twice the 99th percentile.
    </p>
</div>

In [ ]:
print(np.percentile(a, [50, 95, 99]))
print(a.max())

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What if I keep the blank hours, so that hour 5 stays hour 5? I replace each blank with <code>np.nan</code>, short for not a number. It is a float that holds no value, so the array again has all 8,760 places.
    <p>
        Now <code>b.mean()</code> returns <code>nan</code>, because a sum that includes an unknown value is unknown. <code>np.nanmean(b)</code> leaves the <code>nan</code> places out and returns 8.13, and <code>np.isnan(b).sum()</code> counts them: 16, which is 8,760 less 8,744.
    </p>
</div>

In [ ]:
hours = []
for line in lines:
    if line.startswith("RD|"):
        fields = line.split("|")
        if fields[4] == "0002":
            hours.append(float(fields[12]) if fields[12] else np.nan)

b = np.array(hours)
print(len(b), b.mean(), np.nanmean(b), np.isnan(b).sum())

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How many hours were above 35 micrograms per cubic meter? Write <code>b &gt; 35</code> and NumPy compares every hour at once, returning an array of <code>True</code> and <code>False</code>. When you add them up, <code>True</code> counts as 1, so <code>(b &gt; 35).sum()</code> is the count: 36.
    <p>
        Put the same test inside brackets, <code>b[b &gt; 35]</code>, and you get the 36 readings themselves. A <code>nan</code> is not greater than anything, so the 16 blank hours are never counted, and nothing warns you.
    </p>
</div>

In [ ]:
mask = b > 35
print(mask.sum())
print(b[mask])

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 3</span><span style="font-size:1.2em;font-weight:600"> &nbsp;pandas</span>
    <div style="opacity:.92">
        One table for the whole file, eight monitors at once
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Can I skip the loop and have the whole file in one line? <code>pd.read_csv</code> reads it into a table, called a <b>DataFrame</b>, but this file needs hints. <code>sep='|'</code> says the fields are pipe-separated. <code>skiprows=[1]</code> drops the second header line, so the first one becomes the column names. <code>skipfooter=1</code> drops the last line, the one that reads <code>60264 records were written</code>.
    <p>
        Then there is <code>dtype</code>. Left alone, pandas reads site <code>0002</code> as the number 2, and the zeros are gone. Asking for text keeps them.
    </p>
</div>

In [ ]:
df = pd.read_csv(
    path, sep="|", skiprows=[1], skipfooter=1, engine="python",
    dtype={"Site ID": str, "Date": str, "Start Time": str},
)
print(df.shape)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    Did every one of those 60,264 rows get a reading? <code>df['Sample Value']</code> picks out one column, and <code>.describe()</code> summarizes it. The first line, <code>count</code>, says 59,922. The other 342 rows are blank, and <code>describe</code> does not count blanks.
    <p>
        Below the count come the mean, the standard deviation, the minimum, the quartiles and the maximum. The row labeled 50% is the median.
    </p>
</div>

In [ ]:
print(df["Sample Value"].describe())

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do I look at one site? <code>df['Site ID'] == '0002'</code> compares every row and gives True or False, the same kind of mask we used in NumPy. Put it inside <code>df[...]</code> and only the True rows survive. Then pick the column.
    <p>
        The mean of that column is 8.13, the same number the loop and <code>statistics.mean</code> gave. Nobody told pandas to skip the 16 blank hours. It skipped them anyway, and <code>.count()</code> shows how many values it actually used.
    </p>
</div>

In [ ]:
site2 = df[df["Site ID"] == "0002"]["Sample Value"]
print(len(site2), site2.count(), site2.mean())

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How many records does each site have? <code>df['Site ID'].value_counts()</code> tallies them. Six sites have 8,760, which is 365 days of 24 hours. Site 1008 has 5,688 and site 0067 has 2,016, which are 237 days and 84 days, so those two monitors cover only part of the year. Counting rows is the cheapest check there is: a site with fewer rows than the others is a question to ask before you compare it with them.
</div>

In [ ]:
print(df["Site ID"].value_counts())

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What is the mean at each site? Read the line from left to right: <code>df.groupby('Site ID')</code> sorts the rows into one pile per site, <code>['Sample Value']</code> looks at one column in each pile, and <code>.mean()</code> averages each pile. The answer is eight numbers, one per site, from 5.73 at site 0067 to 9.46 at site 1301. The labels down the side are the site IDs, not a column of their own.
</div>

In [ ]:
means = df.groupby("Site ID")["Sample Value"].mean()
print(means)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do I know these readings are PM2.5, and in the units I expect? The file says so itself. The <code>Parameter</code> column holds one value, 88101, and the <code>Unit</code> column holds one value, 105. Those are AQS codes: 88101 is PM2.5, and 105 is micrograms per cubic meter. One value each means nothing is mixed.
    <p>
        Then look at the range. The minimum is 0.3 and the maximum 114.9. Nothing is negative, and nothing looks like a stand-in value such as 9999.
    </p>
</div>

In [ ]:
print(df["Parameter"].unique(), df["Unit"].unique())
print(df["Sample Value"].min(), df["Sample Value"].max())

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0">
    <span style="font-size:1.3em;font-weight:800">Section 4</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Very simple plots</span>
    <div style="opacity:.92">
        One line each: a line, a histogram, a bar chart
    </div>
</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    What does a week of readings look like? The site 0002 values are hourly, so the first 168 of them are seven days. Add <code>.plot()</code> to a pandas column and you get a line chart, with the row number along the bottom and the reading up the side. <code>plt.show()</code> tells matplotlib to draw it. In this week the readings wander between about 0.5 and 16, and nothing stays put for long.
</div>

In [ ]:
site2.head(168).plot()
plt.show()

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How are the readings spread over the whole year? Cut the range into bins and count the hours in each; that picture is a <b>histogram</b>, and <code>.hist(bins=40)</code> draws it. Most hours bunch up between about 2 and 10, and a thin tail runs out to the right, as far as 62. That is why the mean, 8.13, sits above the median, 6.4.
</div>

In [ ]:
site2.hist(bins=40)
plt.show()

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    How do I compare the sites in a picture? Take the <code>means</code> from before, eight numbers with the site as the label, and add <code>.plot.bar()</code>. Each site becomes one bar. Sites 1008 and 0067 have fewer hours than the others, so their bars average a different stretch of the year. The bars run left to right in the order of the site IDs, so the first one is site 0002, at 8.13.
</div>

In [ ]:
means.plot.bar()
plt.show()

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5">
    <div style="font-weight:700;font-size:1.08em;margin-bottom:4px">
        🎯 The vocabulary, at a glance
    </div>
    <ul>
        <li>
            <b>Standard library:</b> <code>open()</code>, <code>.split('|')</code>, <code>float()</code>, and the <code>statistics</code> and <code>math</code> modules. Nothing to install.
        </li>
        <li>
            <b>NumPy arrays:</b> <code>mean</code>, <code>median</code>, <code>std</code>, <code>percentile</code>. <code>ddof=1</code> gives the sample standard deviation.
        </li>
        <li>
            <b>Missing values:</b> <code>np.nan</code> poisons a plain <code>mean</code>; <code>np.nanmean</code> skips it. pandas skips it by default.
        </li>
        <li>
            <b>Masks:</b> <code>b &gt; 35</code> gives True and False. <code>.sum()</code> counts them, and <code>b[mask]</code> selects them.
        </li>
        <li>
            <b>pandas:</b> <code>read_csv</code>, <code>shape</code>, <code>head</code>, <code>dtypes</code>, <code>describe</code>, <code>value_counts</code>, <code>groupby(...).mean()</code>.
        </li>
        <li>
            <b>Sanity checks:</b> row counts against the footer, units and parameter codes, and the minimum and maximum.
        </li>
        <li>
            <b>Plots:</b> <code>.plot()</code>, <code>.hist()</code>, <code>.plot.bar()</code>, then <code>plt.show()</code>.
        </li>
    </ul>
</div>